This file is part of LAO-STO.

Copyright (C) 2025 Julian Czarnecki

This program is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

This program is distributed in the hope that it will be useful,
but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the
GNU General Public License for more details.

You should have received a copy of the GNU General Public License
along with this program.  If not, see <https://www.gnu.org/licenses/>.

If you use this code for scientific research, please cite:
J. Czarnecki et. al.,
"Superconducting gap symmetry of 2DEG at (111)-oriented LaAlO3/SrTiO3 interface",
arXiv:2508.05075 (2025).
https://arxiv.org/abs/2508.05075

In [57]:
import numpy as np
import sympy as sp
from sympy.physics.quantum import TensorProduct

In [58]:
delta_uu, delta_ud, delta_du, delta_dd = sp.symbols(r"\Delta_{\uparrow\uparrow}, \Delta_{\uparrow\downarrow}, \Delta_{\downarrow\uparrow}, \Delta_{\downarrow\downarrow}", complex=True)
delta_pp, delta_pm, delta_mp, delta_mm = sp.symbols(r"\Delta_{++}, \Delta_{+-}, \Delta_{-+}, \Delta_{--}", complex=True)
eps_1, eps_2, eps_3, eps_4 = sp.symbols(r"\epsilon_1, \epsilon_2, \epsilon_3, \epsilon_4")
alpha_R = sp.symbols(r"\alpha_R")
t_hop = sp.symbols(r"t_{hop}")
kx, ky = sp.symbols(r"k_x, k_y", real=True)
eps_vec = [eps_1, eps_2, eps_3, eps_4]

In [59]:
s_x = sp.Matrix([[0, 1], [1, 0]])
s_y = sp.Matrix([[0, -sp.I], [sp.I, 0]])
s_z = sp.Matrix([[1, 0], [0, -1]])
s_0 = sp.Matrix([[1, 0], [0, 1]])


# Simple check without SOC

In [60]:
# Kinetic term (parabolic)
H_kin = ((kx**2 + ky**2) / (2)) * s_0

# Rashba term
H_Rashba = alpha_R * (kx * (-s_y) + ky * s_x)  # alpha*(ky*sigma_x - kx*sigma_y)

# Total Hamiltonian
H_total = H_kin + H_Rashba

# Simplify
H_total = sp.simplify(H_total)

# Display
display(H_total)

Matrix([
[    k_x**2/2 + k_y**2/2, \alpha_R*(I*k_x + k_y)],
[\alpha_R*(-I*k_x + k_y),    k_x**2/2 + k_y**2/2]])

In [61]:
U, H_diag = H_total.diagonalize()
display(sp.simplify(U))
display(H_diag)

theta_k, k = sp.symbols(r"\theta_k k", real=True, positive=True)
polar_subs = {kx: k * sp.cos(theta_k), ky: k * sp.sin(theta_k)}
U_polar = U.subs(polar_subs)
U_polar = sp.simplify(U_polar)
display(U_polar)

Matrix([
[-I*sqrt(k_x**2 + k_y**2)/(k_x + I*k_y), sqrt(-k_x**2 - k_y**2)/(k_x + I*k_y)],
[                                     1,                                    1]])

Matrix([
[-\alpha_R*sqrt(k_x**2 + k_y**2) + k_x**2/2 + k_y**2/2,                                                    0],
[                                                    0, \alpha_R*sqrt(k_x**2 + k_y**2) + k_x**2/2 + k_y**2/2]])

Matrix([
[-I*exp(-I*\theta_k), I*exp(-I*\theta_k)],
[                  1,                  1]])

In [62]:
# Define delta in band basis
Delta_band = sp.Matrix([[delta_pp, delta_pm], [delta_mp, delta_mm]])

U_minus_k = U_polar.subs(theta_k, sp.pi + theta_k)
display(U_minus_k)

Delta_original_basis = U_polar * Delta_band * U_minus_k.T

Delta_original_basis = sp.simplify(Delta_original_basis)
display(Delta_original_basis)

Matrix([
[-I*exp(-I*(\theta_k + pi)), I*exp(-I*(\theta_k + pi))],
[                         1,                         1]])

Matrix([
[(\Delta_{++} - \Delta_{+-} - \Delta_{-+} + \Delta_{--})*exp(-2*I*\theta_k), I*(-\Delta_{++} - \Delta_{+-} + \Delta_{-+} + \Delta_{--})*exp(-I*\theta_k)],
[I*(\Delta_{++} - \Delta_{+-} + \Delta_{-+} - \Delta_{--})*exp(-I*\theta_k),                       \Delta_{++} + \Delta_{+-} + \Delta_{-+} + \Delta_{--}]])

In [63]:
# Integrate over angle to see how periodicity changes the result
Delta_integrated = sp.integrate(Delta_original_basis, (theta_k, 0, 2*sp.pi))
display(Delta_integrated)

Matrix([
[0,                                                            0],
[0, 2*pi*(\Delta_{++} + \Delta_{+-} + \Delta_{-+} + \Delta_{--})]])